In [12]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn.functional as F
import numpy as np
import copy

In [ ]:
#################################
# Path
#################################
dataset_npz_path = "C:/Users/user/Desktop/IDS_masters/dataset/carchallenge_training_0212_231.npz"
test_path = "C:/Users/user/Desktop/IDS_masters/dataset/carchallenge_test_0212_231.npz"
model1_path = "C:/Users/user/Desktop/IDS_masters/model/window128_TCN1.pth"
model2_path = "C:/Users/user/Desktop/IDS_masters/model/window128_TCN2.pth"

epochs = 20
batch_size = 64

Stage1_CH = [0,1,2,3,5,7,8]
Stage2_CH = [0,3,5,6]

device = None

if device is None:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"실행 디바이스: {device}")

실행 디바이스: cuda


In [14]:
#################################
# 1. Dataset Load Class
#################################

class LoadDatset(Dataset):
    def __init__(self, tensor_X, tensor_y):
        # [수정] NumPy 배열이 들어올 경우를 대비한 변환 로직
        if isinstance(tensor_X, np.ndarray):
            tensor_X = torch.from_numpy(tensor_X).float()
        if isinstance(tensor_y, np.ndarray):
            tensor_y = torch.from_numpy(tensor_y).long()

        # [추가] NaN(결측치)을 0.0으로 안전하게 대체하여 학습 붕괴 방지
        self.X = torch.nan_to_num(tensor_X, nan=0.0)
        self.y = tensor_y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    
data_np = np.load(dataset_npz_path)
X_np, y_np = data_np["X"], data_np["y"]

In [15]:
#################################
# 2. Causal Convolution Layer
#################################

class CausalConv1d(nn.Conv1d):
    def __init__(self, in_channels, out_channels, kernel_size, dilation=1):
        super().__init__(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            dilation=dilation,
            padding=0
        )
        self.left_pad = (kernel_size - 1) * dilation

    def forward(self, x):
        x = F.pad(x, (self.left_pad, 0))  # 왼쪽만 패딩
        return super().forward(x)


In [16]:
#################################
# 3. TCN
#################################

class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernal_size=3,  dilation=1, dropout=0.1):
        super().__init__()

        self.conv1 = CausalConv1d(n_inputs, n_outputs, kernel_size=kernal_size, dilation=dilation)
        
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = CausalConv1d(n_outputs, n_outputs, kernel_size=kernal_size, dilation=dilation)
        
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        self.conv3 = CausalConv1d(n_outputs, n_outputs, kernel_size=kernal_size, dilation=dilation)
        
        self.relu3 = nn.ReLU()
        self.dropout3 = nn.Dropout(dropout)
        
      
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()
        self.init_weights()

    def init_weights(self):
        
        def init_one(layer):
            # weight_norm이면 weight_orig가 진짜 파라미터
            w = getattr(layer, "weight_orig", None)
            if w is None:
                w = layer.weight
            nn.init.kaiming_normal_(w)

            if layer.bias is not None:
                nn.init.zeros_(layer.bias)

        # conv1, conv2가 무엇이든(래퍼든 상속이든) 일단 'Conv1d 파라미터 가진 최종 모듈'에 적용
        init_one(self.conv1)
        init_one(self.conv2)
        init_one(self.conv3)

        if self.downsample is not None:
            nn.init.kaiming_normal_(self.downsample.weight)
            if self.downsample.bias is not None:
                nn.init.zeros_(self.downsample.bias)
                
    def forward(self, x):
        # x: (B, C, L)
        out = self.conv1(x)          # CausalConv1d 안에서 패딩 + 오른쪽 잘라내기
        out = self.relu1(out)
        out = self.dropout1(out)

        out = self.conv2(out)
        out = self.relu2(out)
        out = self.dropout2(out)

        out = self.conv3(out)
        out = self.relu3(out)
        out = self.dropout3(out)

        res = x if self.downsample is None else self.downsample(x)
        # CausalConv1d가 길이를 유지하니까 따로 slice 안 해도 됨
        return self.relu(out + res)

In [17]:
#################################
# 4. TCN
#################################

class TemporalConvNet(nn.Module):
    def __init__(self, num_inputs, num_channel, kernel_size=3, dropout =0.2):
        super(TemporalConvNet, self).__init__()
        layers = []
        num_levels = len(num_channel)

        dilation = [1,4,9]

        for i in range(num_levels):
            dilation_size = dilation[i]
            in_channels = num_inputs if i == 0 else num_channel[i-1]
            out_channels = num_channel[i]
            layers += [TemporalBlock(in_channels, out_channels, kernel_size,  dilation = dilation_size, dropout=dropout)]
        self.network = nn.Sequential(*layers)
    def forward(self, x):
        return self.network(x)

In [18]:
#################################
# 5. TCN
#################################

class SeqIDS(nn.Module):
    def __init__(self, num_input, num_classes, dropout_rate=0.2):
        super(SeqIDS, self).__init__()

        self.tcn = TemporalConvNet(
            num_inputs=num_input,          # feature 개수
            num_channel=[32, 64, 128],  # 각 레벨 채널 수
            kernel_size=3,
            dropout=dropout_rate
        )

        self.dropout = nn.Dropout(p=dropout_rate)
        self.classifier = nn.Conv1d(128, num_classes, kernel_size=1)

    def forward(self, x, return_attn=False):
        x = self.tcn(x)
        x = self.dropout(x)
        logits = self.classifier(x)

        if return_attn:
            return logits
        
        return logits

In [19]:
#################################
# 6. Check NAN
#################################

data_load = np.load(dataset_npz_path)
X_np, y_np= data_load["X"], data_load["y"]

### NAN 값 확인 ###
for i in range(X_np.shape[1]):
        feat_nan = np.isnan(X_np[:, i, :]).sum()
        print(f"Feature {i}의 NaN 개수: {feat_nan}")

        nan_count_X = np.isnan(X_np).sum()
        nan_count_y = np.isnan(y_np).sum()
        print(f"데이터 검사 결과:")
        print(f"   - X 내 NaN 개수: {nan_count_X}개")
        print(f"   - y 내 NaN 개수: {nan_count_y}개")
        if nan_count_X > 0:
            print("주의: 입력 데이터(X)의 NaN은 0.0으로 자동 대체되어 학습됩니다.")


inf_count = np.isinf(X_np).sum() # 무한대 체크 추가
print(f"🔍 X 내 inf 개수: {inf_count}개")
if inf_count > 0:
    print("경고: 데이터에 inf(무한대)가 포함되어 있습니다. 인코딩 스크립트를 수정하세요.")

Feature 0의 NaN 개수: 0
데이터 검사 결과:
   - X 내 NaN 개수: 0개
   - y 내 NaN 개수: 0개
Feature 1의 NaN 개수: 0
데이터 검사 결과:
   - X 내 NaN 개수: 0개
   - y 내 NaN 개수: 0개
Feature 2의 NaN 개수: 0
데이터 검사 결과:
   - X 내 NaN 개수: 0개
   - y 내 NaN 개수: 0개
Feature 3의 NaN 개수: 0
데이터 검사 결과:
   - X 내 NaN 개수: 0개
   - y 내 NaN 개수: 0개
Feature 4의 NaN 개수: 0
데이터 검사 결과:
   - X 내 NaN 개수: 0개
   - y 내 NaN 개수: 0개
Feature 5의 NaN 개수: 0
데이터 검사 결과:
   - X 내 NaN 개수: 0개
   - y 내 NaN 개수: 0개
Feature 6의 NaN 개수: 0
데이터 검사 결과:
   - X 내 NaN 개수: 0개
   - y 내 NaN 개수: 0개
Feature 7의 NaN 개수: 0
데이터 검사 결과:
   - X 내 NaN 개수: 0개
   - y 내 NaN 개수: 0개
Feature 8의 NaN 개수: 0
데이터 검사 결과:
   - X 내 NaN 개수: 0개
   - y 내 NaN 개수: 0개
🔍 X 내 inf 개수: 0개


In [20]:
#################################
# 6. First TCN Train
#################################

full_dataset = LoadDatset(X_np, y_np)

#### Validation Split (80:20) ###
total_size = len(full_dataset)
train_size = int(0.8 * total_size)
val_size = total_size - train_size

### 시드 고정 ###
generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=generator)

print(f"데이터 분할 완료: 학습 {train_size}개 / 검증 {val_size}개")

### Model and DataLoader ###
model1 = SeqIDS(num_input=7,num_classes=3, dropout_rate=0.5).to(device)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

### Config Model ###
W_conv1 = model1.tcn.network[0].conv1.weight
other_params = [p for n, p in model1.named_parameters() if "tcn.network.0.conv1.weight" not in n]

optimizer1 = torch.optim.Adam([
    {"params": [W_conv1], "weight_decay": 0},      # conv1은 수동 페널티를 위해 WD 0으로 설정
    {"params": other_params, "weight_decay": 1e-4} # 나머지는 일반적인 WD 적용
    ], lr=1e-4)

scheduler1 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer1, mode='min', factor=0.5, patience=3
)



### data Weight AND LOSS Func ###
weights = torch.tensor([1.0, 1.0, 2.0]).to(device)
criterion1 = nn.CrossEntropyLoss(weight=weights)

### weight decay ###

# 15개 피처에 맞춘 추천 decay_map (Cell 93, 94 공통)
decay_map = {
    0: 0.01,    # ID IAT
    1: 0.01,  # Is Zero ID (중요)
    2: 0.01,    # Raw Entropy
    3: 0.01,   # Hamming Rate (중요)
    4: 0.01,    # Local Freq
    5: 0.01,    # Continuity
    6: 0.01      # Diff Entropy
}


def channel_l2_penalty(conv1_weight, decay_map):
    # conv1_weight: (out_ch, in_ch, k)
    pen = 0.0
    for ch, wd in decay_map.items():   # decay_map: {0:...,1:...,...}
        w_ch = conv1_weight[:, ch:ch+1, :]
        pen = pen + wd * (w_ch.pow(2).sum())
    return pen


# [일반화 5] Early Stopping 변수
best_val_loss = float('inf')
best_model_state = None

prev_train_loss = None
prev_val_loss   = None
overfit_wait    = 0
overfit_patience = 5
min_delta = 1e-4
es_wait   = 0   

print("\n Start Training...")

### Train Loop ###
for epoch in range(1, epochs+ 1):
    model1.train()
    train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        ##====== Slicing Channel ======##
        x1 = X_batch[:, Stage1_CH, :]

        ### y_stage1: 0=OTHER(0/3/4), 1=DoS, 2=Fuzz ###
        y1 = torch.zeros_like(y_batch)
        y1 = torch.where(y_batch == 1, torch.ones_like(y1), y1)
        y1 = torch.where(y_batch == 2, torch.full_like(y1, 2), y1)

        optimizer1.zero_grad()
        
        logits1 = model1(x1)
        loss = criterion1(logits1.permute(0, 2, 1).reshape(-1, 3), y1.reshape(-1))

        W = model1.tcn.network[0].conv1.weight
        loss = loss + channel_l2_penalty(W, decay_map)
        
        loss.backward()
        optimizer1.step()
        train_loss += loss.item()
    
    avg_train_loss = train_loss / len(train_loader)

    # --- Validation ---
    model1.eval()
    val_loss = 0
    with torch.no_grad():
        for X_val, y_val in val_loader:
            X_val, y_val = X_val.to(device), y_val.to(device)

            x1v = X_val[:, Stage1_CH, :]

            ### y_stage1: 0=OTHER(0/3/4), 1=DoS, 2=Fuzz ###
            
            y1v = torch.zeros_like(y_val)
            y1v = torch.where(y_val == 1, torch.ones_like(y1v), y1v)
            y1v = torch.where(y_val == 2, torch.full_like(y1v, 2), y1v)
            
            logits1 = model1(x1v)
            loss = criterion1(logits1.permute(0, 2, 1).reshape(-1, 3), y1v.reshape(-1))
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(val_loader)
    
    scheduler1.step(avg_val_loss)
    current_lr = optimizer1.param_groups[0]['lr']

    print(f"Epoch [{epoch}/{epochs}] Train Loss: {avg_train_loss:.5f} | Val Loss: {avg_val_loss:.5f} | LR: {current_lr:.6f}")

    ### Early Stopping  ###

    improved = (best_val_loss - avg_val_loss) > min_delta
    if improved:
        best_val_loss = avg_val_loss
        best_model_state = copy.deepcopy(model1.state_dict())
        torch.save(best_model_state, model1_path)
        es_wait = 0
    else:
        es_wait += 1
        print(f"  ⚠️ Val Loss 개선 안됨 (ES patience: {es_wait}/{overfit_patience})")
        if es_wait >= overfit_patience:
            print("🛑 Early Stopping 발동! 학습을 조기 종료합니다.")
            break

# 학습 종료 후, 가장 좋았던 모델 상태로 복구
if best_model_state is not None:
    model1.load_state_dict(best_model_state)
    print("\nBest Model 저장")

데이터 분할 완료: 학습 46899개 / 검증 11725개

 Start Training...
Epoch [1/20] Train Loss: 1.52050 | Val Loss: 0.35805 | LR: 0.000100
Epoch [2/20] Train Loss: 0.74309 | Val Loss: 0.22655 | LR: 0.000100
Epoch [3/20] Train Loss: 0.54712 | Val Loss: 0.17701 | LR: 0.000100
Epoch [4/20] Train Loss: 0.42821 | Val Loss: 0.15635 | LR: 0.000100
Epoch [5/20] Train Loss: 0.34228 | Val Loss: 0.14261 | LR: 0.000100
Epoch [6/20] Train Loss: 0.27267 | Val Loss: 0.12066 | LR: 0.000100
Epoch [7/20] Train Loss: 0.20146 | Val Loss: 0.07107 | LR: 0.000100
Epoch [8/20] Train Loss: 0.13619 | Val Loss: 0.05217 | LR: 0.000100
Epoch [9/20] Train Loss: 0.09709 | Val Loss: 0.04292 | LR: 0.000100
Epoch [10/20] Train Loss: 0.07117 | Val Loss: 0.03551 | LR: 0.000100
Epoch [11/20] Train Loss: 0.05325 | Val Loss: 0.02952 | LR: 0.000100
Epoch [12/20] Train Loss: 0.04112 | Val Loss: 0.02606 | LR: 0.000100
Epoch [13/20] Train Loss: 0.03325 | Val Loss: 0.02334 | LR: 0.000100
Epoch [14/20] Train Loss: 0.02854 | Val Loss: 0.02208 | LR:

In [21]:
#################################
# 7. Second TCN Train
#################################

full_dataset = LoadDatset(X_np, y_np)

#### Validation Split (80:20) ###
total_size = len(full_dataset)
train_size = int(0.8 * total_size)
val_size = total_size - train_size

### 시드 고정 ###
generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=generator)

print(f"데이터 분할 완료: 학습 {train_size}개 / 검증 {val_size}개")

### Model and DataLoader ###
model2 = SeqIDS(num_input=4,num_classes=2, dropout_rate=0.5).to(device)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

### Config Model ###
W_conv2 = model2.tcn.network[0].conv1.weight
other_params2 = [p for n, p in model2.named_parameters() if "tcn.network.0.conv1.weight" not in n]

optimizer2 = torch.optim.Adam([
    {"params": [W_conv2], "weight_decay": 0},      # conv1은 수동 페널티를 위해 WD 0으로 설정
    {"params": other_params2, "weight_decay": 1e-4} # 나머지는 일반적인 WD 적용
    ], lr=1e-4)

scheduler2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer2, mode='min', factor=0.5, patience=3
)


### weight decay ###

# Stage 2 전용 (9개 피처에 맞춘 설정)
decay_map = {
    0: 0.01, 1: 0.01, 2: 0.1,
    3: 0.01,  # Z-score Entropy (일반화의 핵심)
    4: 0.01,   # Complexity
    5: 0.01, 6: 0.01,
    7: 0.01, 8: 0.01,
    9: 0.01    # Window ID Entropy (일반화의 핵심)
}


def channel_l2_penalty(conv1_weight, decay_map):
    # conv1_weight: (out_ch, in_ch, k)
    pen = 0.0
    for ch, wd in decay_map.items():   # decay_map: {0:...,1:...,...}
        w_ch = conv1_weight[:, ch:ch+1, :]
        pen = pen + wd * (w_ch.pow(2).sum())
    return pen


# [일반화 5] Early Stopping 변수
best_val_loss = float('inf')
best_model_state = None

prev_train_loss = None
prev_val_loss   = None
overfit_wait    = 0
overfit_patience = 5
min_delta = 1e-4 
es_wait   = 0   

print("\n Start Training...")

### Train Loop ###

for epoch in range(1, epochs+ 1):
    model2.train()
    train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        mask = ((y_batch == 0) | (y_batch == 4)).reshape(-1).float()

        if mask.sum().item() == 0:
            continue

        ### y_stage2: Normal(0), Spoofing(2) ###
        x2 = X_batch[:, Stage2_CH, :]
        y2 = torch.zeros_like(y_batch)
        y2[y_batch == 4] = 1
        y2[(y_batch == 1) | (y_batch == 2) | (y_batch == 3)] = 0
        

        optimizer2.zero_grad()
        
        logits2 = model2(x2)

        class_w = torch.tensor([2.0, 3.0], dtype=torch.float32, device=logits2.device)  # [Normal, Spoof]

        loss_raw = F.cross_entropy(
        logits2.permute(0,2,1).reshape(-1, 2),
        y2.reshape(-1),
        weight=class_w,
        reduction="none"
        )
        ### Loss에 Mask 적용 ###
        den = mask.sum().clamp_min(1.0)
        loss = (loss_raw * mask).sum() / den

        W = model2.tcn.network[0].conv1.weight
        loss = loss + 1e-3 * channel_l2_penalty(W, decay_map)  # 스케일 권장

        loss.backward()
        optimizer2.step()

        train_loss += loss.item()
    
    avg_train_loss = train_loss / len(train_loader)

    # --- Validation ---
    model2.eval()
    val_loss = 0
    with torch.no_grad():
        for X_val, y_val in val_loader:
            X_val, y_val = X_val.to(device), y_val.to(device)
            Bv, Cv, Lv = X_val.shape

           ### y_stage2: Normal(0), Spoofing(2) ###
            x2v = X_val[:, Stage2_CH, :]
            y2v = torch.zeros_like(y_val)
            y2v[y_val == 4] = 1
            y2v[(y_val == 1) | (y_val == 2) | (y_val == 3)] = 0
            mask_v = ((y_val == 0) | (y_val == 4)).reshape(-1).float()
            if mask_v.sum().item() == 0:
                continue
            
            logits2v = model2(x2v)

            loss_raw = F.cross_entropy(
            logits2v.permute(0,2,1).reshape(-1, 2),
            y2v.reshape(-1),
            reduction="none"
            )

            den = mask_v.sum().clamp_min(1.0)
            loss = (loss_raw * mask_v).sum() / den
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(val_loader)
    
    scheduler2.step(avg_val_loss)
    current_lr = optimizer2.param_groups[0]['lr']

    print(f"Epoch [{epoch}/{epochs}] Train Loss: {avg_train_loss:.5f} | Val Loss: {avg_val_loss:.5f} | LR: {current_lr:.6f}")


    ### Early Stopping  ###
    improved = (best_val_loss - avg_val_loss) > min_delta
    if improved:
        best_val_loss = avg_val_loss
        best_model_state2 = copy.deepcopy(model2 .state_dict())
        torch.save(best_model_state2, model2_path)
        es_wait = 0
    else:
        es_wait += 1
        print(f"  ⚠️ Val Loss 개선 안됨 (ES patience: {es_wait}/{overfit_patience})")
        if es_wait >= overfit_patience:
            print("🛑 Early Stopping 발동! 학습을 조기 종료합니다.")
            break

# 학습 종료 후, 가장 좋았던 모델 상태로 복구
if best_model_state2 is not None:
    model2.load_state_dict(best_model_state2)
    print("\nBest Model 저장")

데이터 분할 완료: 학습 46899개 / 검증 11725개

 Start Training...
Epoch [1/20] Train Loss: 0.35518 | Val Loss: 0.12133 | LR: 0.000100
Epoch [2/20] Train Loss: 0.16514 | Val Loss: 0.05892 | LR: 0.000100
Epoch [3/20] Train Loss: 0.13096 | Val Loss: 0.04046 | LR: 0.000100
Epoch [4/20] Train Loss: 0.10727 | Val Loss: 0.03237 | LR: 0.000100
Epoch [5/20] Train Loss: 0.08016 | Val Loss: 0.02128 | LR: 0.000100
Epoch [6/20] Train Loss: 0.04707 | Val Loss: 0.01117 | LR: 0.000100
Epoch [7/20] Train Loss: 0.02806 | Val Loss: 0.00761 | LR: 0.000100
Epoch [8/20] Train Loss: 0.02074 | Val Loss: 0.00627 | LR: 0.000100
Epoch [9/20] Train Loss: 0.01746 | Val Loss: 0.00562 | LR: 0.000100
Epoch [10/20] Train Loss: 0.01543 | Val Loss: 0.00491 | LR: 0.000100
Epoch [11/20] Train Loss: 0.01395 | Val Loss: 0.00454 | LR: 0.000100
Epoch [12/20] Train Loss: 0.01304 | Val Loss: 0.00428 | LR: 0.000100
Epoch [13/20] Train Loss: 0.01230 | Val Loss: 0.00405 | LR: 0.000100
Epoch [14/20] Train Loss: 0.01174 | Val Loss: 0.00386 | LR:

In [23]:
#################################
# 8.  Multi TCN
#################################

model1 = SeqIDS(num_input=7, num_classes=3, dropout_rate=0.5).to(device)
model2 = SeqIDS(num_input=4, num_classes=2, dropout_rate=0.5).to(device)

# 저장된 weight 로드
state1 = torch.load((model1_path), map_location=device)
model1.load_state_dict(state1)

state2 = torch.load((model2_path), map_location=device)
model2.load_state_dict(state2)

model1.eval()
model2.eval()

test_data = np.load(test_path)
X_np, y_np = test_data["X"], test_data["y"]

test_ds = LoadDatset(X_np, y_np)
test_loader = DataLoader(
    test_ds,
    batch_size,
    shuffle=False
)

correct = 0
total = 0
th_dos = 0.9
th_fuzz = 0.3

num_classes = 5
conf_mat = torch.zeros(num_classes, num_classes, dtype=torch.int64)

with torch.no_grad():
    for input, labels in test_loader:
        input = input.to(device)
        labels = labels.to(device)
        #B, C, L = input.permute(0,2,1).shape

        B, C, L = input.shape

        #input = input.permute(0,2,1)
        x1 = input[:, Stage1_CH, :]
        
        ##============= Stage1 ================##
        logit1 = model1(x1)
        p1 = torch.softmax(logit1, dim=1)

        logit_dos = logit1[:,1,:]
        logit_fuzz = logit1[:,2,:]

        dos_mask = p1[:,1,:] >= th_dos
        fuzz_mask = p1[:,2,:] >= th_fuzz

        pred = torch.zeros((B,L), dtype=torch.long, device=device)

        ### dos만 임계값 초과 ###
        only_dos = dos_mask & ~fuzz_mask
        pred[only_dos] = 1

        ### fuzzing만 임계값 초과 ###
        only_fuzz = fuzz_mask & ~dos_mask
        pred[only_fuzz] = 2

        ### dos, fuzzing 모두 임계값 초과 할 때 logit 큰 쪽으로 결과값 ###
        both = dos_mask & fuzz_mask
        pred[both] = torch.where(
            logit_dos[both] >= logit_fuzz[both],
            torch.ones_like(logit_dos[both], dtype=torch.long),   # DoS = 1
            torch.full_like(logit_dos[both], 2, dtype=torch.long) # Fuzz = 2
        )

        ### Stage2로 보낼 위치 ###
        decided = dos_mask | fuzz_mask

        ##============= Stage2 ================##
        x2 = input[:, Stage2_CH, :]
        logit2 = model2(x2)
        pred2 = logit2.argmax(dim=1)

        mapped2 = pred2.clone()
        mapped2[pred2 == 0] = 0 
        mapped2[pred2 == 1] = 4

        pred[~decided] = mapped2[~decided]

        ##============= Metrics =============##
        t_flat = labels.reshape(-1).cpu()
        p_flat = pred.reshape(-1).cpu()

        total += t_flat.numel()
        correct += (t_flat == p_flat).sum().item()

        for t, p in zip(t_flat, p_flat):
            conf_mat[t.long(), p.long()] += 1

accuracy = correct / total

row_sum = conf_mat.sum(dim=1)
tp = conf_mat.diag()
fp = conf_mat.sum(dim=0) - tp
fn = row_sum - tp

precision_per_class = tp / (tp + fp + 1e-12)
recall_per_class    = tp / (tp + fn + 1e-12)
f1_per_class        = 2 * precision_per_class * recall_per_class / (precision_per_class + recall_per_class + 1e-12)

present = row_sum > 0
precision_macro = precision_per_class[present].mean().item()
recall_macro    = recall_per_class[present].mean().item()
f1_macro        = f1_per_class[present].mean().item()

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision(macro, present only): {precision_macro:.4f}")
print(f"Recall(macro, present only)   : {recall_macro:.4f}")
print(f"F1(macro, present only)       : {f1_macro:.4f}")
print("Confusion Matrix:")
print(conf_mat)


EVAL_CLASSES = [0, 1, 2, 4]
eval_idx = torch.tensor(EVAL_CLASSES)

precision_macro = precision_per_class[eval_idx].mean().item()
recall_macro    = recall_per_class[eval_idx].mean().item()
f1_macro        = f1_per_class[eval_idx].mean().item()

LABEL_NAME = {0:"Normal",1:"Dos",2:"Fuzzing",4:"Spoofing"}

print("\n=== Per-class (Attack) Performance ===")
for i in EVAL_CLASSES:
    total_i = int(row_sum[i].item())
    correct_i = int(tp[i].item())
    acc_i = 100.0 * correct_i / total_i if total_i > 0 else 0.0
    print(f"{LABEL_NAME[i]:>10s} : {acc_i:6.2f}%  (correct {correct_i}/{total_i})")

C:\Users\user\AppData\Local\Temp\ipykernel_13776\3330703285.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state1 = torch.load((model1_path), map_location=device)
C:\Us

Accuracy : 0.9752
Precision(macro, present only): 0.7422
Recall(macro, present only)   : 0.7890
F1(macro, present only)       : 0.7633
Confusion Matrix:
tensor([[6662021,       0,   51096,       0,    3083],
        [     16,  383342,       0,       0,       0],
        [   3905,       0,  189470,       0,      11],
        [ 122630,       0,    1667,       0,    1465],
        [   2284,       0,       0,       0,   82882]])

=== Per-class (Attack) Performance ===
    Normal :  99.19%  (correct 6662021/6716200)
       Dos : 100.00%  (correct 383342/383358)
   Fuzzing :  97.98%  (correct 189470/193386)
  Spoofing :  97.32%  (correct 82882/85166)
